In [1]:
!pip install mysql-connector-python

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import pandas as pd
import seaborn as sns
import numpy as np
#Connet to mysql
import mysql.connector

In [3]:
#Creating a link btw mysql and Python

import pyodbc

try:
    conn  = pyodbc.connect(
                            r"DRIVER={ODBC Driver 17 for SQL Server};"
                            r"SERVER=ROHIT\SQLEXPRESS;"
                            r"DATABASE=Advance_DB_Operations;"
                            r"Trusted_Connection=yes;"
                          )
    print("Connection is established")
    
except pyodbc.Error as e:
    print("Conection Failed ", e)
    

Connection is established


In [4]:
cursor = conn.cursor()

cursor.execute("SELECT @@SERVERNAME, DB_NAME()")

print(cursor.fetchone())

('LAPTOP-PHQ3EF2I\\SQLEXPRESS', 'Advance_DB_Operations')


In [5]:
cursor.execute("select COUNT(*) as [Total Suppliers] from suppliers")

In [6]:
row = cursor.fetchone()

In [7]:
row[0]

50

In [8]:
queries = {
    "Total Supplier" : "select COUNT(*) as [Total Suppliers] from suppliers",

    "Total Products" : "select COUNT(product_id) as [Total Products] from products",

    "Total Category Dealing" : "select COUNT(DISTINCT(category)) as [Total Categories Dealing] from products",

    "Total Sales value (Last 3 Months)" : 
    """
                    select ROUND(SUM(ABS(se.change_quantity)*p.price),2)  as [Total sales value in last 3 months]
            from stock_entries as se 
            join
            products as p on 
            p.product_id  = se.product_id
            WHERE se.change_type = 'sale' 
            AND se.entry_date >= (
                                    SELECT DATEADD(MONTH, -3, MAX(entry_date))
                                    FROM stock_entries
                                  );""" ,
    "Total restock value (Last 3 months)" : 
    """
                        select ROUND(SUM(ABS(se.change_quantity)*p.price),2)  as [Total restock value in last 3 months]
            from stock_entries as se 
            join
            products as p on 
            p.product_id  = se.product_id
            WHERE se.change_type = 'Restock' 
            AND se.entry_date >= (
                                    SELECT DATEADD(MONTH, -3, MAX(entry_date))
                                    FROM stock_entries
                                  );""",

    "Below Reorders & No pending reorders" :
    """
                select count(*) as [Below reorder and no pendng records] 
                from products as p 
                where p.stock_quantity<p.reorder_level
                AND product_id NOT IN (select DISTINCT product_id from reorders where status='Pending')  
                """
}

In [9]:
def get_basic_info(cursor):
    queries = {
    "Total Supplier"                    : "select COUNT(*) as [Total Suppliers] from suppliers",

    "Total Products"                    : "select COUNT(product_id) as [Total Products] from products",

    "Total Category Dealing"            : "select COUNT(DISTINCT(category)) as [Total Categories Dealing] from products",

    "Total Sales value (Last 3 Months)" : 
                                         """select ROUND(SUM(ABS(se.change_quantity)*p.price),2)  as [Total sales value in last 3 months]
                                            from stock_entries as se 
                                            join
                                            products as p on 
                                            p.product_id  = se.product_id
                                            WHERE se.change_type = 'sale' 
                                            AND se.entry_date >= (
                                                                    SELECT DATEADD(MONTH, -3, MAX(entry_date))
                                                                    FROM stock_entries
                                                                  );""" ,
        
    "Total restock value (Last 3 months)" : 
                                           """select ROUND(SUM(ABS(se.change_quantity)*p.price),2)  as [Total restock value in last 3 months]
                                              from stock_entries as se 
                                              join
                                              products as p on 
                                              p.product_id  = se.product_id
                                              WHERE se.change_type = 'Restock' 
                                              AND se.entry_date >= (
                                                                    SELECT DATEADD(MONTH, -3, MAX(entry_date))
                                                                    FROM stock_entries
                                                                  );""",

    "Below Reorders & No pending reorders" :
                                             """select count(*) as [Below reorder and no pendng records] 
                                                from products as p 
                                                where p.stock_quantity<p.reorder_level
                                                AND product_id NOT IN (select DISTINCT product_id from reorders where status='Pending')  
                                                """}
    
    result = {}

    for label, query in queries.items():
        cursor.execute(query)
        row = cursor.fetchone()
        result[label] = row[0]

    return result
    
    

In [10]:
queries = {
           "Supplier contact Detauil"         : "select supplier_name, contact_name, email, phone from suppliers",


            "Product with supplier and stock" : """select p.product_name, 
                                                   s.supplier_name, 
                                                   p.stock_quantity, 
                                                   p.reorder_level
                                                    from products as p  
                                                    join 
                                                    suppliers as s 
                                                    on p.supplier_id = s.supplier_id
                                                    order by p.product_name ASC""", 
            
            " Product needing reorder"        : 
                                                   """select product_name, 
                                                   product_id, s
                                                   tock_quantity, 
                                                   eorder_level
                                                    from products 
                                                    where stock_quantity <= reorder_level"""}

In [11]:
tabels = {}

for label, query in queries.items():
    cursor.execute(query)
    tabels[label] = cursor.fetchall()

ProgrammingError: ('42S22', "[42S22] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid column name 's'. (207) (SQLExecDirectW); [42S22] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid column name 'eorder_level'. (207)")

In [ ]:
def get_additional_tabels(cursor):
    queries = {
           "Supplier contact Detauil"         : "select supplier_name, contact_name, email, phone from suppliers",


            "Product with supplier and stock" : """select p.product_name, 
                                                   s.supplier_name, 
                                                   p.stock_quantity, 
                                                   p.reorder_level
                                                    from products as p  
                                                    join 
                                                    suppliers as s 
                                                    on p.supplier_id = s.supplier_id
                                                    order by p.product_name ASC""", 
            
            " Product needing reorder"        : 
                                                   """select product_name, product_id, stock_quantity, reorder_level
                                                    from products 
                                                    where stock_quantity <= reorder_level"""}
    tabels = {}

    for label, query in queries.items():
        cursor.execute(query)
        tabels[label] = cursor.fetchall()

    return tabels
    

In [ ]:
get_additional_tabels(cursor)

In [ ]:
def Add_new_manual_id(cursor, conn, p_name, p_category, p_price, p_stock , p_reorder, p_supplier):
    proc_call = "EXCE AddNewProductManualID ?, ?, ?, ?, ?, ?" 
    params = (p_name, p_category, p_price , p_stock , p_reorder ,p_supplier) 
    cursor.execute(proc_call,  params)

    conn.commit()

In [ ]:
def get_categories(cursor):
    cursor.execute("Select Distinct Category from products order by category asc")
    rows = cursor.fetchall()
    return rows 

In [ ]:
pname = [row[0] for row in get_categories(cursor)]

In [ ]:
pname

In [ ]:
def get_supplier(cursor):
    cursor.execute("Select supplier_id, supplier_name from suppliers order by supplier_name asc")
    rows = cursor.fetchall()
    return rows

In [ ]:

# for (ids, val) in enumerate(get_supplier(cursor)):print(ids, val)

supplier_ids = [row[0] for row in get_supplier(cursor)]
suppliers_name =[row[1] for row in get_supplier(cursor)]


In [ ]:
suppliers_name

In [16]:
def get_pending_reorders(cursor):
    cursor.execute("""Select ro.reorder_id, p.product_name from reorders as ro 
                      join products as p on  
                      ro.product_id = p.product_id""")

    return cursor.fetchall()

In [20]:
[ids [0] for ids in get_pending_reorders(cursor)]

[1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,
 185